# Getting Started with molfp

This notebook walks through the core features of **molfp** — from computing molecular fingerprints and explaining what each bit means, to detecting hash collisions, profiling bioactivity, extracting scaffolds, and building scikit-learn pipelines. Each section introduces a module with a short explanation followed by runnable code.

## 1. Computing Fingerprints

molfp supports 30 molecular fingerprint types across three categories:

- **2D fingerprints** (13): structural keys, circular, path-based, and substructure-based
- **3D fingerprints** (9): shape, electrostatic, and spatial distribution descriptors
- **Graph fingerprints** (8): 4 graph types × base/enriched variants for deep learning

### 1.1 Single Molecule

Pass a SMILES string to any fingerprint function. Returns a list of integers.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*DEPRECATION WARNING.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import molfp
import numpy as np

aspirin = "CC(=O)Oc1ccccc1C(=O)O"

fp = molfp.maccs(aspirin)
print(f"MACCS:   {len(fp)} bits, {sum(fp)} ON")

fp = molfp.ecfp4(aspirin)
print(f"ECFP4:   {len(fp)} bits, {sum(fp)} ON")

fp = molfp.pubchem(aspirin)
print(f"PubChem: {len(fp)} bits, {sum(fp)} ON")

### 1.2 All 30 Fingerprint Types

The `available()` function lists every registered fingerprint with its name, bit count, and description.

In [ ]:
for name in molfp.available():
    fn, n_bits, desc = molfp.get(name)
    print(f"  {name:>30s}  {n_bits:>5d} dims  ({desc})")

### 1.3 Versatile Input

Every fingerprint function accepts multiple input types:

- Single SMILES string → returns a list
- List of SMILES → returns a numpy array
- pandas Series or DataFrame → auto-detects the SMILES column

Use `sparse=True` for memory-efficient output and `n_jobs=-1` for parallel computation.

In [ ]:
smiles = ["CCO", "c1ccccc1", aspirin, "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]

# List input -> numpy array
fps = molfp.ecfp4(smiles)
print(f"List input:  shape={fps.shape}")

# Sparse + parallel
fps_sp = molfp.klekota_roth(smiles, sparse=True, n_jobs=-1)
print(f"Sparse:      shape={fps_sp.shape}, nnz={fps_sp.nnz}")

## 2. Molecule Visualization

molfp provides built-in 2D and 3D molecule rendering that works in Jupyter, Colab, and Streamlit.

In [ ]:
# 2D drawing
img = molfp.draw(smiles, legends=["Ethanol", "Benzene", "Aspirin", "Caffeine"])
img

In [ ]:
# 2D with fingerprint bit highlighting
img = molfp.draw(aspirin, highlight_bits=[88, 143, 162], fp_type="maccs")
img

> **Note:** `molfp.view3d()` requires py3Dmol which works best in local Jupyter. Skipped in Colab.

## 3. Structure Standardization

Clean SMILES before computing fingerprints — desalt, neutralize charges, and canonicalize tautomers.

In [ ]:
cases = [
    ("CC(=O)Oc1ccccc1C(=O)[O-].[Na+]", "sodium aspirin"),
    ("O=C1CC=CC=C1",                    "cyclohexadienone"),
]

for smi, desc in cases:
    clean = molfp.standardize_smiles(smi)
    print(f"  {desc:>20s}: {smi:>40s}  ->  {clean}")

## 4. Bit-Level Explainability & Annotation

`explain()` traces each ON bit back to atoms. `annotate()` gives a chemical name for each bit.

In [ ]:
# Which atoms set which bits?
exp = molfp.explain(aspirin, "ecfp4")
print(f"ECFP4: {len(exp)} ON bits explained")

# Show first 3 bits
for bit, mappings in list(exp.items())[:3]:
    for atom_idx, radius in mappings:
        print(f"  bit {bit}: atom={atom_idx}, radius={radius}")

## 5. Distance & Similarity

Five vectorized pairwise metrics: Tanimoto, Dice, Cosine, Sokal-Michener, and Russel-Rao.

In [ ]:
fp_a = np.array(molfp.maccs("CCO"))
fp_b = np.array(molfp.maccs(aspirin))

print(f"Tanimoto: {molfp.tanimoto(fp_a, fp_b):.3f}")
print(f"Dice:     {molfp.dice(fp_a, fp_b):.3f}")
print(f"Cosine:   {molfp.cosine(fp_a, fp_b):.3f}")

## 6. Hash Collision Detection

Hashed fingerprints can map different substructures to the same bit. Detect and handle these collisions.

In [ ]:
for size in [64, 256, 1024, 4096]:
    coll = molfp.detect_collisions("c1ccc2c(c1)cc1ccc3ccccc3c1c2", "ecfp4", size)
    n_on = sum(molfp.ecfp4("c1ccc2c(c1)cc1ccc3ccccc3c1c2", n_bits=size))
    print(f"  {size:>5d} bits: {len(coll):>2d} collisions / {n_on:>3d} ON bits")

### Handling Collisions

Three strategies: remove colliding bits, use collision-free mode, or flag ambiguous SHAP attributions.

In [ ]:
# Remove bits that have collisions across the dataset
X = np.array(molfp.ecfp4(smiles))
result = molfp.remove_collisions(X, smiles, "ecfp4", 1024)
print(f"Original: {X.shape[1]} bits")
print(f"After removing collisions: {result['n_remaining']} bits (removed {result['n_removed']})")

Collision-free mode uses multi-hash folding, available on all 8 hashed FP types:

In [ ]:
# Standard vs collision-free
fp_std = molfp.ecfp4(aspirin)
fp_cf = molfp.ecfp4(aspirin, collision_free=True)
print(f"Standard ECFP4: {sum(fp_std)} bits ON")
print(f"Collision-free: {sum(fp_cf)} bits ON (more bits due to multi-hash)")

## 7. Bioactivity Profiling

Classify molecules by activity threshold and identify which fingerprint bits are enriched in active compounds.

In [ ]:
# Sample data
smi = ["CCO", "c1ccccc1", aspirin, "OC(=O)c1ccccc1O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
       "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "c1ccncc1", "CC(=O)O"]
act = [4.5, 5.0, 7.2, 6.8, 5.5, 7.0, 4.0, 3.5]

# Classify by activity threshold
labels = molfp.classify(act, metric="pchembl")
print(f"Labels: {labels}")
print(f"Active: {labels.count('active')}, Inactive: {labels.count('inactive')}")

## 8. Activity Cliffs

Find molecule pairs that are structurally similar but have large activity differences.

In [ ]:
cliffs = molfp.activity_cliffs(smi, act, "maccs", sim_threshold=0.3, activity_threshold=1.0)
print(f"Found {len(cliffs)} activity cliffs")
if len(cliffs) > 0:
    print(f"  First cliff: {cliffs[0]}")

## 9. Consensus / Fusion Fingerprints

Combine multiple fingerprint types by selecting the most informative bits from each.

In [ ]:
result = molfp.fuse(smi, act, fp_types=["maccs", "estate", "pubchem"], n_bits=20, method="variance")
print(f"Fused FP: {result['fused_fps'].shape}")

# Which FP types contributed which bits?
for i, (fp_name, bit) in list(result["source_map"].items())[:5]:
    print(f"  fused bit {i} <- {fp_name} bit {bit}")

## 10. SHAP Interpretation with Chemical Meaning

Train a classifier, compute SHAP values, and map the top features back to chemical substructures.

In [ ]:
result = molfp.interpret(smi, act, fp_type="maccs", metric="pchembl")
print(f"Model accuracy: {result['accuracy']:.2f}")
print(f"Labels: {result['label_set']}")

# Top features by SHAP importance
for feat in result["feature_importance"][:5]:
    print(f"  bit {feat['bit']}: SHAP={feat['mean_abs_shap']:.4f}, {feat['chemical_meaning']}")

## 11. Feature Selection

Remove constant features and select the most informative ones by mutual information.

In [ ]:
# Compute a high-dimensional fingerprint
X = np.array(molfp.klekota_roth(smi))
labels = molfp.classify(act, metric="pchembl")
y = [1 if l == "active" else 0 for l in labels]

# Remove near-constant features
result = molfp.remove_low_variance(X, threshold=0.01)
print(f"Original: {X.shape[1]} -> After variance filter: {result['X'].shape[1]}")

## 12. Molecular Graph Representations

Convert molecules to atom/bond feature matrices for graph neural networks. Four graph types are available.

In [ ]:
# Standard graph: atoms=nodes, bonds=edges
g = molfp.mol_graph("c1ccccc1")
print(f"Atom features: {g['atom_features'].shape}")
print(f"Edge index:    {g['edge_index'].shape}")
print(f"Edge features: {g['edge_features'].shape}")

In [ ]:
# All 4 graph types
for t in ["standard", "distance", "line", "fragment"]:
    g = molfp.mol_graph(aspirin, type=t)
    print(f"  {t:>10s}: {g['atom_features'].shape[0]:>3d} nodes, {g['edge_index'].shape[1]:>3d} edges")

In [ ]:
# Graph vectors for sklearn (fixed-length)
X = molfp.graph_vectors(smi, type="standard", enriched=True)
print(f"Graph vectors: {X.shape}")

## 13. scikit-learn Integration

Drop-in transformers for any sklearn pipeline. One per fingerprint type, plus `GraphTransformer`.

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.ensemble import RandomForestClassifier

# Single FP pipeline
pipe = make_pipeline(molfp.ECFPTransformer(), RandomForestClassifier(n_estimators=10))
print(f"Pipeline: {pipe}")

## 14. Scaffold Analysis

Extract Murcko scaffolds with chemical names from ~130 common ring systems.

In [ ]:
# Extract scaffolds with chemical names
drugs = [
    "CC(=O)Oc1ccccc1C(=O)O",           # Aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",    # Caffeine
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",       # Ibuprofen
    "c1ccc2c(c1)cc1ccc3ccccc3c1c2",      # Anthracene
]

for smi in drugs:
    scaf = molfp.scaffold(smi)
    if scaf:
        print(f"  {scaf['name']:>20s}: {scaf['murcko']}")

### Scaffold-Aware Train/Test Split

Split molecules so that no scaffold appears in both train and test — prevents data leakage.

In [ ]:
# Scaffold-aware split — no leakage
result = molfp.scaffold_split(smi, test_size=0.2)
print(f"Train: {len(result['train_idx'])} molecules")
print(f"Test:  {len(result['test_idx'])} molecules")

## Summary

| Category | Count | Key Functions |
|----------|-------|---------------|
| **2D Fingerprints** | 13 | `maccs()`, `ecfp4()`, `pubchem()`, `klekota_roth()`, ... |
| **3D Fingerprints** | 9 | `usr()`, `usrcat()`, `whim()`, `getaway()`, ... |
| **Graph Fingerprints** | 8 | `mol_graph()`, `graph_vectors()`, `to_pyg()`, `to_dgl()` |
| **Visualization** | — | `draw()`, `view3d()` |
| **Standardize** | — | `preprocess()`, `standardize_smiles()` |
| **Explain** | — | `explain()`, `annotate()` |
| **Interpret** | — | `interpret()`, `interpret_molecule()`, `compare_molecules()` |
| **Feature Selection** | — | `select_features()`, `remove_low_variance()` |
| **Distance** | — | `tanimoto()`, `pairwise()` — 5 vectorized metrics |
| **Collisions** | — | `detect_collisions()`, `remove_collisions()`, `collision_free=True` |
| **Bioactivity** | — | `classify()`, `enriched_bits()` |
| **Cliffs** | — | `activity_cliffs()`, `cliff_summary()` |
| **Fusion** | — | `fuse()`, `complementarity()` |
| **Scaffold** | — | `scaffold()`, `scaffold_fingerprint()`, `scaffold_split()` |
| **sklearn** | — | 22 transformers + `GraphTransformer` |